# Llama 3.2 3B Instruct — QLoRA malware-family classification (7 classes)
**Split:** your existing 60/20/20 JSONL files (17,693 / 5,898 / 5,898) — unchanged.
**Training recipe = the earlier Llama-3.2 vulnerability run** (Trainer, chat-template prompt with prompt tokens masked to -100, QLoRA 4-bit NF4 + double quant, LoRA r16/α32/dropout 0.05 on all 7 projections, lr 2e-4, cosine, warmup 100, fp16, paged AdamW 8-bit, batch 1 × grad-accum 16, 1 epoch, seed 42, gradient checkpointing, no eval during training).
**Hardware:** both T4s via DDP (`NUM_GPUS = 2`), effective batch kept at 16. **Only data-driven differences:** max length 2,816 (malware prompts are ~2,700 tokens; 768 would cut off the label), system+user messages from your malware JSONL, 7 classes, and checkpoints **every 50 steps → Google Drive** with auto-resume.
**Evaluation (same convention as the earlier Llama eval):** greedy decoding, label extraction, UNKNOWN rate reported; metrics strict (UNKNOWN = wrong) **and** valid-only; per-example CSV. Additionally split into rows whose feature vector does / does not appear in train.

### How to use
| What you want | Set `RUN_MODE` | Then |
|---|---|---|
| Train (first time **or** resume) | `"train"` | **Save Version → Save & Run All (Commit)**. Re-commit until the log says `TRAINING COMPLETE`. |
| Validate the final adapter | `"eval_val"` | Commit again (also runs automatically after training if time is left). |
| Final test (once, after model selection) | `"eval_test"` + `CONFIRM_FINAL_MODEL_SELECTED = True` | Commit. |

Commit mode keeps running after you close the browser (up to 12 h); interactive sessions die when idle.

### One-time setup (before the first run)
1. **Llama licence:** accept it on huggingface.co/meta-llama/Llama-3.2-3B-Instruct, then create a *read* token.
2. **Kaggle secrets** (Add-ons → Secrets, attach both to this notebook): `HF_TOKEN`, `RCLONE_CONFIG_B64` (see next cell).
3. **Data:** upload your `train.jsonl`, `validation.jsonl`, `test.jsonl` (the `messages` format) as a Kaggle Dataset and add it as input. File names must start with `train`, `val`, `test`.
4. **Settings:** Accelerator = GPU T4 ×2 (both used), Internet = On, Persistence = Files only (optional).

### Creating `RCLONE_CONFIG_B64` (on your laptop, once)
```bash
# 1. install rclone: https://rclone.org/downloads/   (Windows: winget install Rclone.Rclone)
rclone config
#   n (new remote) -> name: gdrive -> storage: drive -> client_id/secret: Enter (blank)
#   scope: 1 (full drive) -> root_folder_id / service_account: Enter -> advanced: n
#   auto config: y (browser opens, sign in to Google) -> shared drive: n -> y (keep) -> q
rclone lsd gdrive:            # should list your Drive folders

# 2. base64 the config and paste the single line into the Kaggle secret RCLONE_CONFIG_B64
base64 -w0 ~/.config/rclone/rclone.conf          # Linux
base64 -i ~/.config/rclone/rclone.conf           # macOS
# Windows PowerShell:
# [Convert]::ToBase64String([IO.File]::ReadAllBytes("$env:APPDATA\rclone\rclone.conf"))
```
The remote **must be named `gdrive`**. If you already have this secret from your Qwen/DeepSeek runs, reuse it.

In [1]:
# ===================== 1. RUN CONTROL =====================
import os, sys, time, subprocess, importlib
SESSION_START = time.time()                  # used by the 11.25 h time guard

RUN_MODE = "train"                           # "train" | "eval_val" | "eval_test"
NUM_GPUS = 2                                 # 2 = DDP on both T4s (one full model copy per GPU), 1 = single GPU.
                                             # Keep the same value for the whole training run.
AUTO_VALIDATE_AFTER_TRAINING = True          # in "train" mode: run validation if training finishes in this session
CONFIRM_FINAL_MODEL_SELECTED = False         # must be True for "eval_test"

TRAIN_ENV = {"EVAL_SUBSET": "0",             # 0 = no eval during training (as in earlier runs); e.g. "300" for eval loss
             "EVAL_STEPS": "200",
             "MAX_RUNTIME_HOURS": "11.25"}   # save + stop before Kaggle's 12 h limit

# If auto-detection of your JSONL files fails, set explicit paths:
# os.environ["SRC_TRAIN"] = "/kaggle/input/<dataset>/train.jsonl"
# os.environ["SRC_VALIDATION"] = "/kaggle/input/<dataset>/validation.jsonl"
# os.environ["SRC_TEST"] = "/kaggle/input/<dataset>/test.jsonl"

def run_script(cmd, extra_env=None):
    # Run a script in a subprocess that sees ONLY physical GPU 0 (prevents DataParallel)
    env = os.environ.copy()
    env.update({"CUDA_VISIBLE_DEVICES": "0", "PYTHONUNBUFFERED": "1", "SESSION_START": str(SESSION_START),
                "TOKENIZERS_PARALLELISM": "false", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
    env.update(extra_env or {})
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"{os.path.basename(cmd[-1])} exited with code {rc} - read the log above")
print("RUN_MODE =", RUN_MODE)

RUN_MODE = train


## 2. Shared config + Drive helpers

In [2]:
%%writefile /kaggle/working/common.py
# common.py - shared config + Google Drive (rclone) helpers for the Llama 3.2 malware experiment
import os, re, json, time, glob, shutil, hashlib, subprocess

# ----------------------------- experiment constants -----------------------------
MODEL_ID    = "meta-llama/Llama-3.2-3B-Instruct"
SEED        = 42
MAX_LENGTH  = 2816
# The Llama 3.x chat template writes "Today Date: <today>" into the system header.
# Pinning it keeps every prompt identical across sessions (train / resume / eval).
DATE_STRING = "26 Jul 2024"
EOT         = "<|eot_id|>"

LABELS = ["Benign", "RedLineStealer", "Downloader", "RAT", "BankingTrojan", "SnakeKeyLogger", "Spyware"]
EXPECTED_COUNTS = {"train": 17693, "validation": 5898, "test": 5898}

# ----------------------------- local paths -----------------------------
WORK_DIR   = "/kaggle/working"
DATA_DIR   = f"{WORK_DIR}/llama32_malware_data"
OUTPUT_DIR = f"{WORK_DIR}/llama32-malware-output"
FINAL_DIR  = f"{OUTPUT_DIR}/final_adapter"
EVAL_DIR   = f"{WORK_DIR}/llama32-malware-eval"

# ----------------------------- Google Drive (rclone) -----------------------------
# Kept outside /kaggle/working so the secret never ends up in saved notebook output.
RCLONE_CONF  = os.path.expanduser("~/.config/rclone/rclone.conf")
REMOTE_ROOT  = "gdrive:llama32-malware-backups/llama32-training"
REMOTE_CKPT  = f"{REMOTE_ROOT}/checkpoints"
REMOTE_FINAL = f"{REMOTE_ROOT}/final_adapter"
REMOTE_EVAL  = f"{REMOTE_ROOT}/eval"
REMOTE_LOGS  = f"{REMOTE_ROOT}/logs"
KEEP_REMOTE_CHECKPOINTS = 3
UPLOAD_MARKER        = "UPLOAD_COMPLETE"     # written last -> checkpoint on Drive is whole
TRAINING_DONE_MARKER = "TRAINING_COMPLETE"   # written after the final adapter is on Drive

COPY_FLAGS = ["--transfers", "4", "--checkers", "8", "--retries", "5",
              "--low-level-retries", "20", "--drive-chunk-size", "64M"]
LOCAL_REQUIRED = ["adapter_model.safetensors", "trainer_state.json", "optimizer.pt", "scheduler.pt"]


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def rclone(*args, check=True):
    cmd = ["rclone", "--config", RCLONE_CONF, *args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"rclone {' '.join(args)} failed (exit {r.returncode}):\n{r.stderr[-2000:]}")
    return r


def _not_found(r):
    err = r.stderr.lower()
    return r.returncode == 3 or "directory not found" in err or "object not found" in err


def rclone_query(*args, tries=5):
    """Read-only rclone call that RETRIES transient Drive errors and RAISES if they persist.
    A genuine 'not found' is returned normally (caller treats it as empty)."""
    for i in range(tries):
        r = rclone(*args, check=False)
        if r.returncode == 0 or _not_found(r):
            return r
        log(f"rclone {args[0]} {args[1] if len(args) > 1 else ''} failed (try {i + 1}/{tries}, exit {r.returncode}): "
            f"{r.stderr.strip()[-400:]}")
        time.sleep(10 * (i + 1))
    raise RuntimeError(f"Google Drive is not answering correctly (rclone {' '.join(args)}). "
                       f"Refusing to continue so checkpoints are not ignored. Re-run the notebook.")


def remote_exists(path):
    r = rclone_query("lsf", path)
    return r.returncode == 0 and r.stdout.strip() != ""


def upload_dir(local, remote):
    rclone("copy", local, remote, *COPY_FLAGS)


def upload_file(local, remote_path):
    rclone("copyto", local, remote_path, *COPY_FLAGS)


def write_remote_marker(remote_path):
    tmp = "/tmp/_marker.txt"
    with open(tmp, "w") as f:
        f.write(time.strftime("%Y-%m-%d %H:%M:%S"))
    rclone("copyto", tmp, remote_path)


def training_complete():
    return remote_exists(f"{REMOTE_FINAL}/{TRAINING_DONE_MARKER}")


# ----------------------------- checkpoints -----------------------------
def ckpt_step(name):
    m = re.fullmatch(r"checkpoint-(\d+)/?", name.strip())
    return int(m.group(1)) if m else None


def list_remote_checkpoints():
    """Steps of checkpoints on Drive whose upload finished (marker present)."""
    r = rclone_query("lsf", REMOTE_CKPT, "-R", "--files-only", "--include", f"/checkpoint-*/{UPLOAD_MARKER}")
    if r.returncode != 0:          # folder genuinely does not exist yet
        return []
    steps = {ckpt_step(line.split("/")[0]) for line in r.stdout.splitlines() if "/" in line}
    return sorted(s for s in steps if s is not None)


def is_complete_local(path):
    return all(os.path.exists(os.path.join(path, f)) for f in LOCAL_REQUIRED)


def list_local_checkpoints():
    steps = []
    for p in glob.glob(f"{OUTPUT_DIR}/checkpoint-*"):
        s = ckpt_step(os.path.basename(p))
        if s is not None and is_complete_local(p):
            steps.append(s)
    return sorted(steps)


def prune_remote_checkpoints(keep=KEEP_REMOTE_CHECKPOINTS):
    done = list_remote_checkpoints()
    for s in done[:-keep]:
        rclone("purge", f"{REMOTE_CKPT}/checkpoint-{s}", "--drive-use-trash=false", check=False)
        log(f"Drive: removed old checkpoint-{s}")
    # half-uploaded folders (session died mid-upload) older than the newest good one
    if done:
        r = rclone("lsf", REMOTE_CKPT, "--dirs-only", check=False)
        for line in r.stdout.splitlines():
            s = ckpt_step(line)
            if s is not None and s not in done and s < done[-1]:
                rclone("purge", f"{REMOTE_CKPT}/checkpoint-{s}", "--drive-use-trash=false", check=False)
                log(f"Drive: removed incomplete checkpoint-{s}")


def upload_checkpoint(local_path):
    name = os.path.basename(local_path.rstrip("/"))
    dst = f"{REMOTE_CKPT}/{name}"
    t = time.time()
    rclone("copy", local_path, dst, *COPY_FLAGS)
    rclone("check", local_path, dst, "--one-way", "--size-only")   # every local file is on Drive
    write_remote_marker(f"{dst}/{UPLOAD_MARKER}")
    log(f"Drive: {name} uploaded + verified in {time.time() - t:.0f}s -> {dst}")
    prune_remote_checkpoints()


def restore_latest_checkpoint():
    """Return a local checkpoint path to resume from (downloading from Drive if needed), or None."""
    local, remote = list_local_checkpoints(), list_remote_checkpoints()
    best_local = local[-1] if local else -1
    best_remote = remote[-1] if remote else -1
    log(f"Checkpoints - local: {local or 'none'} | Drive: {remote or 'none'}")
    if best_local < 0 and best_remote < 0:
        # safety net: never silently start from scratch while checkpoint folders exist on Drive
        r = rclone_query("lsf", REMOTE_CKPT, "--dirs-only")
        dirs = [d for d in r.stdout.split() if ckpt_step(d) is not None] if r.returncode == 0 else []
        if dirs:
            raise RuntimeError(f"Drive has checkpoint folders {dirs} but none is usable (no {UPLOAD_MARKER}). "
                               f"Refusing to restart from scratch - check Drive, then re-run.")
        return None
    if best_remote > best_local:
        dst = f"{OUTPUT_DIR}/checkpoint-{best_remote}"
        shutil.rmtree(dst, ignore_errors=True)
        log(f"Downloading checkpoint-{best_remote} from Drive ...")
        rclone("copy", f"{REMOTE_CKPT}/checkpoint-{best_remote}", dst, *COPY_FLAGS)
        marker = os.path.join(dst, UPLOAD_MARKER)
        if os.path.exists(marker):
            os.remove(marker)
        if not is_complete_local(dst):
            raise RuntimeError(f"Downloaded checkpoint is missing files: {os.listdir(dst)}")
        return dst
    path = f"{OUTPUT_DIR}/checkpoint-{best_local}"
    if best_local > best_remote:           # saved locally but never reached Drive
        try:
            upload_checkpoint(path)
        except Exception as e:
            log(f"WARNING: could not back up {path}: {e}")
    return path


# ----------------------------- data helpers -----------------------------
_SPLIT_PREFIX = {"train": "train", "validation": "val", "test": "test"}


def find_source_jsonl(split):
    """The original conversational JSONL ({"messages": [...]}) for a split, under /kaggle/input.
    Override with env var SRC_TRAIN / SRC_VALIDATION / SRC_TEST if auto-detection is ambiguous."""
    override = os.environ.get(f"SRC_{split.upper()}")
    if override:
        return override
    pre = _SPLIT_PREFIX[split]
    cands = sorted(p for p in glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
                   if os.path.basename(p).lower().startswith(pre)
                   and "prompt_completion" not in os.path.basename(p).lower())
    if len(cands) != 1:
        raise FileNotFoundError(f"Expected one {split} JSONL under /kaggle/input, found: {cands}. "
                                f"Set os.environ['SRC_{split.upper()}'] to the right path.")
    return cands[0]


def read_jsonl(path):
    with open(path) as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def label_of(record):
    return record["messages"][2]["content"].strip()


def feature_hash(record):
    return hashlib.md5(record["messages"][1]["content"].encode()).hexdigest()


def render_prompt(tokenizer, messages):
    """[system, user] -> Llama 3 chat text ending with the assistant header (starts with <|begin_of_text|>)."""
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                         date_string=DATE_STRING)


def parse_label(text):
    t = text.strip().split("\n")[0].strip().strip(".").strip()
    lower = {l.lower(): l for l in LABELS}
    if t.lower() in lower:
        return lower[t.lower()]
    for l in sorted(LABELS, key=len, reverse=True):   # longest first so "RedLineStealer" beats "RAT"
        if t.lower().startswith(l.lower()):
            return l
    return "UNKNOWN"

Writing /kaggle/working/common.py


## 3. Packages

In [3]:
import os, sys, base64, shutil, subprocess, importlib
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
sys.path.insert(0, "/kaggle/working")
import common; importlib.reload(common); from common import *

secrets = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"), add_to_git_credential=False)   # cached in ~/.cache, not in /kaggle/working

if shutil.which("rclone") is None:
    subprocess.run("curl -fsSL https://downloads.rclone.org/rclone-current-linux-amd64.zip -o /tmp/rclone.zip && "
                   "unzip -q -o /tmp/rclone.zip -d /tmp/rclone && "
                   "cp /tmp/rclone/rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone",
                   shell=True, check=True)
os.makedirs(os.path.dirname(RCLONE_CONF), exist_ok=True)
with open(RCLONE_CONF, "wb") as f:
    f.write(base64.b64decode(secrets.get_secret("RCLONE_CONFIG_B64")))
os.chmod(RCLONE_CONF, 0o600)

print(rclone("version").stdout.splitlines()[0])
rclone("mkdir", REMOTE_CKPT)
write_remote_marker(f"{REMOTE_LOGS}/drive_write_test.txt")
assert remote_exists(f"{REMOTE_LOGS}/drive_write_test.txt"), "Drive write test failed"
print("Drive OK ->", REMOTE_ROOT)
print("Complete checkpoints on Drive:", list_remote_checkpoints() or "none")
print("Training complete flag:", training_complete())

rclone v1.76.0
Drive OK -> gdrive:llama32-malware-backups/llama32-training
Complete checkpoints on Drive: [800, 850, 900]
Training complete flag: False


In [ ]:
import importlib.util, subprocess, sys
need = ["transformers", "peft", "bitsandbytes", "accelerate", "datasets", "sklearn"]
missing = [m for m in need if importlib.util.find_spec(m) is None]
if missing:
    pkgs = [("scikit-learn" if m == "sklearn" else m) for m in missing]
    print("Installing:", pkgs)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
import torch, transformers, peft, bitsandbytes, accelerate, datasets
for m in [torch, transformers, peft, bitsandbytes, accelerate, datasets]:
    print(f"{m.__name__:14s} {m.__version__}")
print("GPUs visible to the notebook:", torch.cuda.device_count(), "(training subprocess will see only 1)")

## 4. Secrets → Hugging Face login + rclone (Google Drive)

## 5. Data: messages → prompt/completion (train + validation only)
Uses your existing split files — nothing is re-split. The test file is **not** converted or loaded here.
Prompts are rendered with the Llama 3.2 chat template (date pinned), the completion is `"<Label><|eot_id|>"`, and every example is checked to fit in 2,816 tokens.

In [4]:
import os, sys, json, collections, importlib, numpy as np
from transformers import AutoTokenizer
sys.path.insert(0, "/kaggle/working")
import common; importlib.reload(common); from common import *

def prepare_data():
    os.makedirs(DATA_DIR, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    assert tok.convert_tokens_to_ids(EOT) == 128009
    report = {}
    train_hashes = {feature_hash(r) for r in read_jsonl(find_source_jsonl("train"))}
    for split in ["train", "validation"]:
        out = f"{DATA_DIR}/{split}_prompt_completion.jsonl"
        src = find_source_jsonl(split)
        rows = []
        for r in read_jsonl(src):
            msgs = r["messages"]
            assert [m["role"] for m in msgs] == ["system", "user", "assistant"]
            y = label_of(r)
            assert y in LABELS, y
            rows.append({"prompt": render_prompt(tok, msgs[:2]), "completion": y + EOT})
        assert len(rows) == EXPECTED_COUNTS[split], (split, len(rows))
        assert rows[0]["prompt"].startswith("<|begin_of_text|>") and DATE_STRING in rows[0]["prompt"]
        lens = []
        for k in range(0, len(rows), 500):
            batch = [x["prompt"] + x["completion"] for x in rows[k:k + 500]]
            lens += [len(ids) for ids in tok(batch, add_special_tokens=False)["input_ids"]]
        lens = np.array(lens)
        assert lens.max() <= MAX_LENGTH, f"{split}: {lens.max()} tokens > {MAX_LENGTH}"
        with open(out, "w") as f:
            for x in rows:
                f.write(json.dumps(x) + "\n")
        counts = collections.Counter(x["completion"].replace(EOT, "") for x in rows)
        report[split] = {"source": src, "n": len(rows), "class_counts": dict(counts),
                         "tokens": {"min": int(lens.min()), "mean": float(lens.mean()), "p99": float(np.percentile(lens, 99)),
                                    "max": int(lens.max())}}
        print(f"{split:10s} {len(rows):6d} rows | tokens min {lens.min()} mean {lens.mean():.0f} max {lens.max()} | {dict(counts)}")
    for split in ["validation", "test"]:
        recs = list(read_jsonl(find_source_jsonl(split)))
        seen = sum(feature_hash(r) in train_hashes for r in recs)
        report[f"{split}_rows_with_identical_feature_vector_in_train"] = seen
        print(f"{split}: {seen}/{len(recs)} rows ({seen / len(recs):.1%}) have a feature vector identical to a train row")
    with open(f"{DATA_DIR}/split_report.json", "w") as f:
        json.dump(report, f, indent=2)
    upload_file(f"{DATA_DIR}/split_report.json", f"{REMOTE_LOGS}/split_report.json")
    print("Example prompt tail:", repr(rows[0]["prompt"][-120:]))
    print("Example completion :", repr(rows[0]["completion"]))

if RUN_MODE == "train" and not training_complete():
    if all(os.path.exists(f"{DATA_DIR}/{s}_prompt_completion.jsonl") for s in ["train", "validation"]):
        print("Prompt-completion files already exist - skipping conversion.")
    else:
        prepare_data()

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

train       17693 rows | tokens min 2631 mean 2656 max 2710 | {'RAT': 2973, 'Benign': 1126, 'RedLineStealer': 3013, 'Downloader': 2786, 'BankingTrojan': 3046, 'SnakeKeyLogger': 2530, 'Spyware': 2219}
validation   5898 rows | tokens min 2631 mean 2656 max 2710 | {'RAT': 991, 'Benign': 376, 'Downloader': 928, 'BankingTrojan': 1015, 'RedLineStealer': 1004, 'SnakeKeyLogger': 844, 'Spyware': 740}
validation: 1298/5898 rows (22.0%) have a feature vector identical to a train row
test: 1278/5898 rows (21.7%) have a feature vector identical to a train row
Example prompt tail: 'oRawData: 0\npdata_Characteristics: 0\n\nPredicted Malware Family:<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
Example completion : 'RAT<|eot_id|>'


## 6. Training script (written to disk, run in a single-GPU subprocess)

In [5]:
%%writefile /kaggle/working/train_llama32_single.py
# train_llama32_single.py - Llama 3.2 3B Instruct + QLoRA, 1 epoch, 1 GPU or 2-GPU DDP (NUM_GPUS), Drive checkpoint every 50 steps
import os, sys, json, time

WORLD_SIZE = int(os.environ.get("WORLD_SIZE", "1"))     # 1 = single GPU, 2 = DDP via torchrun (one process per GPU)
LOCAL_RANK = int(os.environ.get("LOCAL_RANK", "0"))
IS_MAIN = LOCAL_RANK == 0
_vis = os.environ.get("CUDA_VISIBLE_DEVICES", "")
if not _vis or len(_vis.split(",")) != WORLD_SIZE:
    sys.exit(f"Run this through the launch cell. CUDA_VISIBLE_DEVICES={_vis!r} must list exactly {WORLD_SIZE} GPU(s). "
             "Never plain DataParallel.")

sys.path.insert(0, "/kaggle/working")
from common import *  # noqa: E402,F401

import torch  # noqa: E402
from datasets import load_dataset  # noqa: E402
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,  # noqa: E402
                          Trainer, TrainingArguments, TrainerCallback, set_seed)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training  # noqa: E402

# ----------------------------- protocol (do not change without a reason) -----------------------------
EPOCHS        = 1
BATCH_SIZE    = 1
EFFECTIVE_BATCH = 16                       # protocol: 16 samples per optimizer step
assert EFFECTIVE_BATCH % WORLD_SIZE == 0
GRAD_ACCUM    = EFFECTIVE_BATCH // (BATCH_SIZE * WORLD_SIZE)   # 16 on 1 GPU, 8 per GPU on 2 GPUs
LR            = 2e-4
WARMUP_STEPS  = 100
SAVE_STEPS    = 50
LORA_CFG = dict(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

# ----------------------------- run controls (set by the launch cell) -----------------------------
EVAL_SUBSET       = int(os.environ.get("EVAL_SUBSET", "0"))     # earlier runs: eval_strategy="no". >0 = optional eval-loss subset
EVAL_STEPS        = int(os.environ.get("EVAL_STEPS", "200"))
MAX_RUNTIME_HOURS = float(os.environ.get("MAX_RUNTIME_HOURS", "11.25"))
SESSION_START     = float(os.environ.get("SESSION_START", time.time()))
LOG_FILE          = f"{OUTPUT_DIR}/training_log.jsonl"
RUN_SETUP_REMOTE  = f"{REMOTE_LOGS}/run_setup.json"


def mlog(msg):
    if IS_MAIN:
        log(msg)


def barrier():
    if WORLD_SIZE > 1:
        torch.distributed.barrier()


class DriveBackupCallback(TrainerCallback):
    """After every checkpoint save: upload to Drive, verify, mark complete, keep newest 3."""
    def __init__(self):
        self.failures = 0

    def on_save(self, args, state, control, **kwargs):
        barrier()                                   # every rank finished writing its part of the checkpoint
        if not state.is_world_process_zero:
            return
        path = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")
        if not os.path.isdir(path):
            log(f"WARNING: expected {path} after save, not found")
            return
        try:
            upload_checkpoint(path)
            if os.path.exists(LOG_FILE):
                upload_file(LOG_FILE, f"{REMOTE_LOGS}/training_log.jsonl")
            self.failures = 0
        except Exception as e:  # never kill training because Drive hiccuped; next save retries
            self.failures += 1
            log(f"!!! Drive upload FAILED ({self.failures} in a row) - training continues: {e}")


class TimeGuardCallback(TrainerCallback):
    """Save + stop cleanly before Kaggle's 12 h limit kills the session."""
    def __init__(self):
        self.fired = False

    def on_step_end(self, args, state, control, **kwargs):
        hours = (time.time() - SESSION_START) / 3600
        stop = torch.tensor([1 if hours >= MAX_RUNTIME_HOURS else 0], device=f"cuda:{LOCAL_RANK}")
        if WORLD_SIZE > 1:
            torch.distributed.broadcast(stop, src=0)    # all ranks stop on the same step (no deadlock)
        if stop.item() == 1 and not self.fired:
            self.fired = True
            mlog(f"Time guard: {hours:.2f} h used - saving checkpoint-{state.global_step} and stopping. "
                f"Re-run the notebook to resume.")
            control.should_save = True
            control.should_training_stop = True
        return control


class ProgressCallback(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs):
        self.t0, self.step0 = time.time(), state.global_step

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not state.is_world_process_zero:
            return
        rec = {k: v for k, v in (logs or {}).items()}
        rec.update(step=state.global_step, max_steps=state.max_steps, epoch=state.epoch,
                   wall=time.strftime("%Y-%m-%d %H:%M:%S"),
                   gpu_peak_gb=round(torch.cuda.max_memory_allocated() / 2**30, 2))
        done = state.global_step - self.step0
        eta = ""
        if done > 0:
            sps = (time.time() - self.t0) / done
            rec["sec_per_step"] = round(sps, 1)
            eta = f" | {sps:.0f}s/step | ETA {(state.max_steps - state.global_step) * sps / 3600:.1f} h (total, all sessions)"
        with open(LOG_FILE, "a") as f:
            f.write(json.dumps(rec) + "\n")
        parts = [f"{k}={v:.4g}" if isinstance(v, float) else f"{k}={v}" for k, v in (logs or {}).items()]
        log(f"step {state.global_step}/{state.max_steps} | " + " ".join(parts) +
            f" | peak {rec['gpu_peak_gb']} GB{eta}")


def main():
    from accelerate import PartialState
    assert torch.cuda.is_available() and torch.cuda.device_count() == WORLD_SIZE, \
        f"Expected {WORLD_SIZE} visible GPU(s), got {torch.cuda.device_count()}"
    torch.cuda.set_device(LOCAL_RANK)
    state = PartialState()                      # initialises the process group under torchrun
    set_seed(SEED)
    mlog(f"GPUs: {WORLD_SIZE} x {torch.cuda.get_device_name(0)} | per-GPU batch {BATCH_SIZE} x grad-accum {GRAD_ACCUM} "
         f"x {WORLD_SIZE} GPU(s) = effective batch {BATCH_SIZE * GRAD_ACCUM * WORLD_SIZE} | torch {torch.__version__}")

    done = [training_complete() if IS_MAIN else False]      # only rank 0 talks to Drive
    if WORLD_SIZE > 1:
        torch.distributed.broadcast_object_list(done, src=0)
    if done[0]:
        mlog("Drive already has TRAINING_COMPLETE - final adapter exists. Nothing to train.")
        return
    if IS_MAIN:
        rclone("mkdir", REMOTE_CKPT)
        os.makedirs(OUTPUT_DIR, exist_ok=True)
    barrier()

    # ----------------------------- tokenizer + data -----------------------------
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    files = {"train": f"{DATA_DIR}/train_prompt_completion.jsonl",
             "validation": f"{DATA_DIR}/validation_prompt_completion.jsonl"}
    with state.main_process_first():
        raw = load_dataset("json", data_files=files)
    assert len(raw["train"]) == EXPECTED_COUNTS["train"], len(raw["train"])
    assert len(raw["validation"]) == EXPECTED_COUNTS["validation"], len(raw["validation"])

    def tokenize(ex):
        # Same masking as the earlier Llama/Qwen runs: tokenize prompt and full text, mask the prompt part with -100.
        # Prompt already contains <|begin_of_text|> from the chat template -> add_special_tokens=False (no double BOS)
        prompt_ids = tok(ex["prompt"], add_special_tokens=False)["input_ids"]
        full_ids = tok(ex["prompt"] + ex["completion"], add_special_tokens=False)["input_ids"]
        labels = list(full_ids)
        for i in range(min(len(prompt_ids), len(labels))):
            labels[i] = -100                      # loss only on the completion ("RAT<|eot_id|>")
        return {"input_ids": full_ids, "labels": labels, "n_tokens": len(full_ids)}

    def target_check(e):
        return any(l != -100 for l in e["labels"])

    with state.main_process_first():           # rank 0 tokenizes, rank 1 reuses the cache
        ds = raw.map(tokenize, remove_columns=raw["train"].column_names, num_proc=2, desc="Tokenizing")
    train_ds = ds["train"]
    max_len = max(train_ds["n_tokens"])
    assert max_len <= MAX_LENGTH, f"Longest example {max_len} > MAX_LENGTH {MAX_LENGTH}"
    ex = train_ds[0]
    assert ex["input_ids"][0] == tok.bos_token_id and ex["input_ids"].count(tok.bos_token_id) == 1
    assert target_check(ex), "completion tokens not masked correctly"
    target = tok.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])
    mlog(f"Train {len(train_ds)} | longest {max_len} tokens (limit {MAX_LENGTH}) | example target: {target!r}")

    eval_ds = None
    if EVAL_SUBSET > 0:
        eval_ds = ds["validation"].shuffle(seed=SEED).select(range(EVAL_SUBSET))
        mlog(f"Periodic eval loss on fixed {EVAL_SUBSET}-sample validation subset every {EVAL_STEPS} steps")

    def collate(features):                      # same as earlier CausalLMCollator (pad to multiple of 8)
        m = max(len(f["input_ids"]) for f in features)
        m = ((m + 7) // 8) * 8
        pad = tok.pad_token_id
        return {
            "input_ids": torch.tensor([f["input_ids"] + [pad] * (m - len(f["input_ids"])) for f in features]),
            "labels": torch.tensor([f["labels"] + [-100] * (m - len(f["labels"])) for f in features]),
            "attention_mask": torch.tensor([[1] * len(f["input_ids"]) + [0] * (m - len(f["input_ids"])) for f in features]),
        }

    # ----------------------------- model: 4-bit NF4 + LoRA -----------------------------
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": LOCAL_RANK},          # each process keeps a full copy on its own GPU
                                                 dtype=torch.float16)
    model.config.use_cache = False
    model.config.pretraining_tp = 1
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
    model = get_peft_model(model, LoraConfig(**LORA_CFG))
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    mlog(f"Trainable {trainable:,} / {total:,} ({100 * trainable / total:.4f}%) | "
        f"VRAM after load {torch.cuda.memory_allocated() / 2**30:.2f} GB")

    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="cosine",
        # bitsandbytes 8-bit AdamW. Steps 1-400 ran with the *paged* variant (same update rule); the paged
        # variant crashes after a checkpoint resume (cudaMemPrefetchAsync on non-managed memory,
        # "invalid argument ... pythonInterface.cpp:578"), so the non-paged version is used. State loads 1:1.
        optim="adamw_bnb_8bit",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        fp16=True,                       # as in the earlier Llama / Qwen / DeepSeek runs
        logging_steps=10,
        logging_first_step=True,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        eval_strategy="steps" if eval_ds is not None else "no",
        eval_steps=EVAL_STEPS,
        prediction_loss_only=True,       # never gather 128k-vocab logits during eval
        seed=SEED,
        data_seed=SEED,
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=2,
        ddp_find_unused_parameters=False,
        disable_tqdm=True,
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                      data_collator=collate, processing_class=tok,
                      callbacks=[ProgressCallback(), DriveBackupCallback(), TimeGuardCallback()])

    # ----------------------------- train (auto-resume) -----------------------------
    resume_path = None                                  # None -> fresh start (never "")
    if IS_MAIN:
        resume_path = restore_latest_checkpoint()
        r = rclone("cat", RUN_SETUP_REMOTE, check=False)
        prev = json.loads(r.stdout) if r.returncode == 0 and r.stdout.strip() else ({"world_size": 1} if resume_path else None)
        if resume_path and prev["world_size"] != WORLD_SIZE:
            raise SystemExit(f"Checkpoints on Drive were made with {prev['world_size']} GPU(s) but this run uses {WORLD_SIZE}. "
                             f"Set NUM_GPUS = {prev['world_size']} in cell 1 (switching mid-run changes the data order), "
                             f"or delete the Drive checkpoints to start over.")
        if not resume_path:
            tmp = "/tmp/run_setup.json"
            with open(tmp, "w") as f:
                json.dump({"world_size": WORLD_SIZE, "grad_accum": GRAD_ACCUM, "started": time.strftime("%Y-%m-%d %H:%M:%S")}, f)
            upload_file(tmp, RUN_SETUP_REMOTE)
        expect = int(os.environ.get("EXPECT_RESUME_STEP", "0"))
        got = ckpt_step(os.path.basename(resume_path)) if resume_path else 0
        if expect and got < expect:
            raise SystemExit(f"The notebook saw checkpoint-{expect} on Drive but the script would resume from "
                             f"{resume_path or 'scratch'}. Stopping instead of losing progress - re-run the notebook.")
    if WORLD_SIZE > 1:
        obj = [resume_path]
        torch.distributed.broadcast_object_list(obj, src=0)
        resume_path = obj[0]
    mlog(f"Resume from: {resume_path or 'fresh start'}")
    result = trainer.train(resume_from_checkpoint=resume_path)

    st = trainer.state
    if st.global_step < st.max_steps:
        mlog(f"Stopped at step {st.global_step}/{st.max_steps}. Latest checkpoint is on Drive - "
            f"re-run the notebook (RUN_MODE='train') to continue.")
        return

    # ----------------------------- final artifacts -----------------------------
    trainer.save_model(FINAL_DIR)            # LoRA adapter only (called on all ranks; rank 0 writes)
    barrier()
    if not IS_MAIN:
        return
    tok.save_pretrained(FINAL_DIR)
    import transformers, peft, bitsandbytes, datasets
    run_config = dict(
        model_id=MODEL_ID, seed=SEED, max_length=MAX_LENGTH, date_string=DATE_STRING,
        epochs=EPOCHS, batch_size_per_gpu=BATCH_SIZE, grad_accum_per_gpu=GRAD_ACCUM, num_gpus=WORLD_SIZE,
        effective_batch=BATCH_SIZE * GRAD_ACCUM * WORLD_SIZE,
        lr=LR, warmup_steps=WARMUP_STEPS, scheduler="cosine",
        optimizer="bitsandbytes 8-bit AdamW (paged_adamw_8bit for steps 1-400, adamw_bnb_8bit after resume; identical update rule)",
        quantization="4-bit NF4, double quant, fp16 compute", lora=LORA_CFG,
        trainer_fp16=True, gradient_checkpointing=True,
        loss="completion-only (prompt tokens masked with -100)",
        train_samples=len(train_ds), eval_subset=EVAL_SUBSET, global_steps=st.global_step,
        trainable_params=trainable, total_params=total,
        versions=dict(torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,
                      bitsandbytes=bitsandbytes.__version__, datasets=datasets.__version__),
        last_session_metrics=result.metrics,
    )
    with open(f"{FINAL_DIR}/run_config.json", "w") as f:
        json.dump(run_config, f, indent=2)
    with open(f"{FINAL_DIR}/training_history.json", "w") as f:
        json.dump(st.log_history, f, indent=1)
    if os.path.exists(LOG_FILE):
        shutil.copy(LOG_FILE, f"{FINAL_DIR}/training_log.jsonl")
    upload_dir(FINAL_DIR, REMOTE_FINAL)
    write_remote_marker(f"{REMOTE_FINAL}/{TRAINING_DONE_MARKER}")
    log(f"TRAINING COMPLETE - {st.global_step} steps. Final adapter on Drive: {REMOTE_FINAL}")


if __name__ == "__main__":
    main()

Writing /kaggle/working/train_llama32_single.py


## 7. Evaluation script (generated labels → metrics; resumable)

In [6]:
%%writefile /kaggle/working/eval_generate.py
# eval_generate.py - classification metrics from GENERATED labels (not LM loss). Resumable via Drive.
import os, sys, json, time, argparse, collections

if os.environ.get("CUDA_VISIBLE_DEVICES") != "0":
    sys.exit("Run this through the notebook cell (it sets CUDA_VISIBLE_DEVICES=0).")
sys.path.insert(0, "/kaggle/working")
from common import *  # noqa: E402,F401

ap = argparse.ArgumentParser()
ap.add_argument("--split", choices=["validation", "test"], required=True)
ap.add_argument("--adapter", default=FINAL_DIR,
                help="local adapter dir, or a Drive path like gdrive:.../checkpoints/checkpoint-1100")
ap.add_argument("--tag", default="final", help="name for this evaluation run (results folder)")
ap.add_argument("--batch-size", type=int, default=8)
ap.add_argument("--limit", type=int, default=0, help="evaluate only the first N samples (quick check)")
a = ap.parse_args()

if a.split == "test" and os.environ.get("ALLOW_TEST_EVAL") != "1":
    sys.exit("Test evaluation is locked. Finish validation + model selection first, then use the test cell.")

import torch  # noqa: E402
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig  # noqa: E402
from peft import PeftModel  # noqa: E402

run_name = f"{a.tag}_{a.split}" + (f"_first{a.limit}" if a.limit else "")
out_dir = f"{EVAL_DIR}/{run_name}"
remote_dir = f"{REMOTE_EVAL}/{run_name}"
preds_path = f"{out_dir}/predictions.jsonl"
os.makedirs(out_dir, exist_ok=True)

# ----------------------------- adapter -----------------------------
adapter = a.adapter
if adapter.startswith("gdrive:"):
    local = f"{WORK_DIR}/adapters/{adapter.rstrip('/').split('/')[-1]}"
    if not os.path.exists(f"{local}/adapter_config.json"):
        log(f"Downloading adapter {adapter}")
        rclone("copy", adapter, local, *COPY_FLAGS)
    adapter = local
elif not os.path.exists(f"{adapter}/adapter_config.json") and adapter == FINAL_DIR:
    log("Final adapter not found locally - downloading from Drive")
    rclone("copy", REMOTE_FINAL, FINAL_DIR, *COPY_FLAGS)
assert os.path.exists(f"{adapter}/adapter_config.json"), f"No adapter at {adapter}"

# ----------------------------- data -----------------------------
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token = tok.eos_token
tok.padding_side = "left"          # left padding for batched generation

records = list(read_jsonl(find_source_jsonl(a.split)))
assert len(records) == EXPECTED_COUNTS[a.split], len(records)
train_hashes = {feature_hash(r) for r in read_jsonl(find_source_jsonl("train"))}
if a.limit:
    records = records[:a.limit]

# resume: pull partial predictions from Drive if this session has none
if not os.path.exists(preds_path):
    rclone("copy", remote_dir, out_dir, "--include", "predictions.jsonl", check=False)
done = {}
if os.path.exists(preds_path):
    for r in read_jsonl(preds_path):
        done[r["idx"]] = r
pending = [i for i in range(len(records)) if i not in done]
log(f"{a.split}: {len(records)} samples | already predicted {len(done)} | to do {len(pending)} | adapter {adapter}")

# ----------------------------- model -----------------------------
if pending:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map={"": 0},
                                                 dtype=torch.float16)
    model = PeftModel.from_pretrained(model, adapter)
    model.eval()
    stop_ids = list({tok.convert_tokens_to_ids(EOT), tok.eos_token_id})

    bs, i, t0, n_new, last_upload = a.batch_size, 0, time.time(), 0, 0
    with open(preds_path, "a") as fout:
        while i < len(pending):
            idx = pending[i:i + bs]
            prompts = [render_prompt(tok, records[j]["messages"][:2]) for j in idx]
            enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
            try:
                with torch.inference_mode():
                    out = model.generate(**enc, max_new_tokens=8, do_sample=False, temperature=None, top_p=None,
                                         eos_token_id=stop_ids, pad_token_id=tok.pad_token_id)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                log(f"OOM - batch size -> {bs}")
                continue
            texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            for j, txt in zip(idx, texts):
                rec = {"idx": j, "true": label_of(records[j]), "pred": parse_label(txt), "raw": txt,
                       "seen_in_train": feature_hash(records[j]) in train_hashes}
                done[j] = rec
                fout.write(json.dumps(rec) + "\n")
            fout.flush()
            i += len(idx)
            n_new += len(idx)
            if n_new - last_upload >= 200 or i >= len(pending):
                last_upload = n_new
                rate = (time.time() - t0) / n_new
                acc = sum(r["true"] == r["pred"] for r in done.values()) / len(done)
                log(f"{len(done)}/{len(records)} | running acc {acc:.4f} | {rate:.2f}s/sample | "
                    f"ETA {(len(pending) - i) * rate / 60:.0f} min")
                try:
                    upload_file(preds_path, f"{remote_dir}/predictions.jsonl")
                except Exception as e:
                    log(f"WARNING: Drive upload failed (will retry): {e}")

# ----------------------------- metrics -----------------------------
import numpy as np  # noqa: E402
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,  # noqa: E402
                             classification_report, confusion_matrix)

rows = [done[i] for i in range(len(records))]


def metrics(subset):
    if not subset:
        return None
    y = [r["true"] for r in subset]
    p = [r["pred"] for r in subset]
    res = {"n": len(subset), "accuracy": accuracy_score(y, p),
           "n_unknown": sum(x == "UNKNOWN" for x in p)}
    for avg in ["macro", "weighted"]:
        pr, rc, f1, _ = precision_recall_fscore_support(y, p, labels=LABELS, average=avg, zero_division=0)
        res[f"{avg}_precision"], res[f"{avg}_recall"], res[f"{avg}_f1"] = pr, rc, f1
    res["unknown_rate"] = res["n_unknown"] / len(subset)
    res["per_class"] = classification_report(y, p, labels=LABELS, output_dict=True, zero_division=0)
    # same convention as the earlier Llama vulnerability eval: also report metrics on parseable outputs only
    v = [(t, q) for t, q in zip(y, p) if q != "UNKNOWN"]
    if v:
        yv, pv = zip(*v)
        res["valid_only"] = {"n": len(v), "accuracy": accuracy_score(yv, pv)}
        for avg in ["macro", "weighted"]:
            pr, rc, f1, _ = precision_recall_fscore_support(yv, pv, labels=LABELS, average=avg, zero_division=0)
            res["valid_only"][f"{avg}_precision"], res["valid_only"][f"{avg}_recall"], res["valid_only"][f"{avg}_f1"] = pr, rc, f1
    return res


results = {
    "split": a.split, "adapter": a.adapter, "tag": a.tag,
    "all": metrics(rows),
    "unseen_feature_vectors": metrics([r for r in rows if not r["seen_in_train"]]),
    "seen_feature_vectors": metrics([r for r in rows if r["seen_in_train"]]),
}
y_true = [r["true"] for r in rows]
y_pred = [r["pred"] for r in rows]
cols = LABELS + ["UNKNOWN"]
cm = confusion_matrix(y_true, y_pred, labels=cols)[:len(LABELS)]   # rows: true classes; cols incl. UNKNOWN

with open(f"{out_dir}/metrics.json", "w") as f:
    json.dump(results, f, indent=2)
with open(f"{out_dir}/classification_report.txt", "w") as f:
    f.write(classification_report(y_true, y_pred, labels=LABELS, digits=4, zero_division=0))
with open(f"{out_dir}/confusion_matrix.csv", "w") as f:
    f.write("true\\pred," + ",".join(cols) + "\n")
    for lab, row in zip(LABELS, cm):
        f.write(lab + "," + ",".join(map(str, row)) + "\n")

import matplotlib  # noqa: E402
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
fig, ax = plt.subplots(figsize=(9, 7))
norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right")
ax.set_yticks(range(len(LABELS)), LABELS)
for r in range(cm.shape[0]):
    for c in range(cm.shape[1]):
        ax.text(c, r, cm[r, c], ha="center", va="center", fontsize=8,
                color="white" if norm[r, c] > 0.5 else "black")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Llama 3.2 3B QLoRA - {a.split} (n={len(rows)}, acc={results['all']['accuracy']:.4f})")
fig.tight_layout()
fig.savefig(f"{out_dir}/confusion_matrix.png", dpi=150)

import csv  # noqa: E402
with open(f"{out_dir}/eval_per_example.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["index", "true_label", "predicted_label", "generated_text", "seen_in_train"])
    for r in rows:
        w.writerow([r["idx"], r["true"], r["pred"], r["raw"], r["seen_in_train"]])

upload_dir(out_dir, remote_dir)
m = results["all"]
log(f"{a.split.upper()} | acc {m['accuracy']:.4f} | macro F1 {m['macro_f1']:.4f} | weighted F1 {m['weighted_f1']:.4f} "
    f"| UNKNOWN outputs {m['n_unknown']} ({m['unknown_rate']:.2%})")
u = results["unseen_feature_vectors"]
if u:
    log(f"  unseen-in-train feature vectors (n={u['n']}): acc {u['accuracy']:.4f} | macro F1 {u['macro_f1']:.4f}")
log(f"Results saved to {out_dir} and {remote_dir}")

Writing /kaggle/working/eval_generate.py


## 8. TRAIN — starts fresh or resumes from the newest checkpoint on Drive
With `NUM_GPUS = 2` this runs DDP: each T4 holds its own 4-bit model copy and processes half the samples; effective batch stays 16 (1 × 8 × 2), so 1,106 steps, same lr schedule. The log prints **seconds/step and total ETA** — if it needs more than one 12-hour session, just re-commit; it continues from Drive. Don't change `NUM_GPUS` in the middle of a run (the script refuses).

In [ ]:
import sys
if RUN_MODE == "train":
    sys.path.insert(0, "/kaggle/working")
    from common import list_remote_checkpoints, training_complete
    seen = list_remote_checkpoints()
    print("Checkpoints on Drive before launch:", seen or "none (fresh start)")
    TRAIN_ENV["EXPECT_RESUME_STEP"] = str(seen[-1]) if seen else "0"   # script must resume from at least this step
    script = "/kaggle/working/train_llama32_single.py"
    if NUM_GPUS > 1:   # torchrun: one process per GPU, each pinned to its own GPU (DDP) - not DataParallel
        cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={NUM_GPUS}", script]
    else:
        cmd = [sys.executable, script]
    run_script(cmd, {**TRAIN_ENV,
                     "CUDA_VISIBLE_DEVICES": ",".join(str(i) for i in range(NUM_GPUS)),
                     "NCCL_P2P_DISABLE": "1", "NCCL_IB_DISABLE": "1"})   # stable NCCL on Kaggle T4 x2
    if AUTO_VALIDATE_AFTER_TRAINING and training_complete():
        print("\nTraining complete - starting validation evaluation (resumable if the session ends).")
        run_script([sys.executable, "/kaggle/working/eval_generate.py", "--split", "validation", "--tag", "final"])

Checkpoints on Drive before launch: [800, 850, 900]

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[16:27:48] GPUs: 2 x Tesla T4 | per-GPU batch 1 x grad-accum 8 x 2 GPU(s) = effective batch 16 | torch 2.11.0+cu130
/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)

Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 4508 examples [00:00, 25958.37 examples/s]
Generating train split: 10517 examples [00:00, 29986.52 examples/s]
Generating train split: 13523 examples [00:00, 29672.46 examples/s]
Generating train split: 17693 

## 9. VALIDATION (full 5,898) — for model selection
Optional: evaluate an intermediate checkpoint too, e.g. `--adapter gdrive:llama32-malware-backups/llama32-training/checkpoints/checkpoint-1100 --tag ckpt1100`.
Quick smoke check first: add `"--limit", "64"`.

In [ ]:
import os, sys
if RUN_MODE == "eval_val":
    run_script([sys.executable, "/kaggle/working/eval_generate.py", "--split", "validation", "--tag", "final"])

def show_results(run_name):
    import json
    from IPython.display import Image, display
    d = f"{EVAL_DIR}/{run_name}"
    if not os.path.exists(f"{d}/metrics.json"):
        rclone("copy", f"{REMOTE_EVAL}/{run_name}", d, check=False)
    if not os.path.exists(f"{d}/metrics.json"):
        print("No results yet for", run_name); return
    m = json.load(open(f"{d}/metrics.json"))
    for k in ["all", "unseen_feature_vectors", "seen_feature_vectors"]:
        if m[k]:
            x = m[k]
            print(f"{k:24s} n={x['n']:5d} acc={x['accuracy']:.4f} macroP={x['macro_precision']:.4f} "
                  f"macroR={x['macro_recall']:.4f} macroF1={x['macro_f1']:.4f} weightedF1={x['weighted_f1']:.4f} unknown={x['n_unknown']}")
    print(open(f"{d}/classification_report.txt").read())
    display(Image(f"{d}/confusion_matrix.png"))

if RUN_MODE in ("train", "eval_val"):
    show_results("final_validation")

## 10. TEST (5,898) — run once, only after validation-based model selection

In [ ]:
import sys
if RUN_MODE == "eval_test":
    assert CONFIRM_FINAL_MODEL_SELECTED, "Set CONFIRM_FINAL_MODEL_SELECTED = True only after you've finished model selection on validation."
    run_script([sys.executable, "/kaggle/working/eval_generate.py", "--split", "test", "--tag", "final"],
               {"ALLOW_TEST_EVAL": "1"})
    show_results("final_test")

## Troubleshooting
| Symptom | Cause / fix |
|---|---|
| `FileNotFoundError: Expected one train JSONL` | Set `SRC_TRAIN` / `SRC_VALIDATION` / `SRC_TEST` in cell 1. |
| `rclone ... failed` / `invalid_grant` | Drive token revoked/expired → rerun `rclone config reconnect gdrive:` on your laptop and update the secret. |
| `401` / `gated repo` | Licence not accepted on the HF account that made `HF_TOKEN`. |
| OOM during training | Check `peak` in the log (≈8.5 GB/GPU expected). If it persists, set `NUM_GPUS = 1` and restart from scratch (delete Drive checkpoints). |
| 2-GPU run hangs at the start (no step logs for >15 min) | NCCL issue → set `NUM_GPUS = 1` and start over. |
| `Attempting to unscale FP16 gradients` | Set `fp16=False` in the training script (the handover's fallback) and note it in the report. |
| `Drive upload FAILED` lines | Training continues; the next save (50 steps later) retries. Repeated failures → check the secret / Drive quota. |
| Stopped by time guard | Normal. Re-commit with `RUN_MODE="train"`; it downloads the newest checkpoint and continues. |

**Drive layout:** `llama32-malware-backups/llama32-training/` → `checkpoints/checkpoint-N/` (newest 3, each with `UPLOAD_COMPLETE`), `logs/`, `final_adapter/` (+`TRAINING_COMPLETE`), `eval/final_validation/`, `eval/final_test/`.